## Score Candidates

In [ ]:
import json
from pathlib import Path
from typing import Literal

from langchain_anthropic import ChatAnthropic
from langchain_core.caches import InMemoryCache
from langchain_core.globals import set_llm_cache
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

from recruit import Candidates

# Set global cache to in-memory
set_llm_cache(InMemoryCache())

In [ ]:
class Skills(BaseModel):
    skill: str = Field(
        ...,
        description="Short two to three word description of the skill",
    )
    description: str = Field(
        ...,
        description="A concise description of the skill",
    )


class SkillSet(BaseModel):
    required: list[Skills] = Field(
        ...,
        description="A list of required skills",
    )
    nice_to_have: list[Skills] = Field(
        ...,
        description="A list of nice-to-have skills",
    )

In [ ]:
llm = ChatOpenAI(model="gpt-5.6-terra")
skill_agent = llm.with_structured_output(
    SkillSet,
    method="json_schema",
)

In [ ]:
openenings = Path("..") / "openings"
# job_description_path = openenings / "frontend-engineer-berlin.md"
job_description_path = openenings / "technical-founders-associate-berlin.md"
# job_description_path = openenings / "ai-success-engineer-berlin.md"
# job_description_path = openenings / "president-and-coo-london-berlin.md"

with open(job_description_path, "r") as file:
    job_description = file.read()

In [ ]:
system_prompt = f"""
Extract skills from the job description, do not list more than 6 skills.

Guidelines:
- All skills must be mutually exclusive and collectively exhaustive
- Required skills are non-negotiable for the role, while nice-to-have skills are optional but beneficial
""".strip()

In [ ]:
messages = [
    ("system", system_prompt),
    ("human", job_description),
]

raw = skill_agent.invoke(messages)
skills = SkillSet.model_validate(raw)

In [ ]:
print(skills.model_dump_json(indent=2))

In [ ]:
class SkillScore(Skills):
    match: bool = Field(
        ...,
        description="Indicates whether the candidate possesses the skill",
    )
    match_degree: Literal["low", "medium", "high"] = Field(
        ...,
        description="Indicates the degree to which the candidate possesses the skill",
    )
    match_notes: str = Field(
        ...,
        description="A brief explanation of the match degree",
    )


class CandidateScore(BaseModel):
    required: list[SkillScore] = Field(
        ...,
        description="A list of required skills with match information",
    )

    nice_to_have: list[SkillScore] = Field(
        ...,
        description="A list of nice-to-have skills with match information",
    )

In [ ]:
score_prompt = f"""
Task:
- Judge the candidate's skills against the extracted skills.
- Indicate whether the candidate possesses it and the degree of match (low, medium, high).
- Only consider the skills extracted from the job description. The job description serves as context. Do not add any additional skills.

Skill Set:
```json
{skills.model_dump_json(indent=2)}
```
""".strip()

In [ ]:
candidates = Candidates()
resume = candidates.get_person(485761352)

In [ ]:
score_agent = llm.with_structured_output(
    CandidateScore,
    method="json_schema",
)

messages = [
    ("system", score_prompt),
    # ("human", job_description),
    # ("ai", skills.model_dump_json(indent=2)),
    ("human", json.dumps(resume, indent=2)),
]

raw = score_agent.invoke(messages)
candidate_score = CandidateScore.model_validate(raw)

In [ ]:
print(candidate_score.model_dump_json(indent=2))